# Member 1: full spatial pipeline

The existing 01_dataset_exploration.ipynb and data_loader.py remain unchanged. Use src.member1_io.load_trip(..., load_reference=False) for inference. Features never use .route. Development labels are derived by monotone route-position alignment, not independently verified point labels. Arc IDs are local to trips.

Outputs in results/member1/data: candidates.parquet, observations.parquet, folds.csv, feature_config.json, inventory.csv and content-hashed per-trip caches. Share all four schema/config files with M2/M3. Join by (split, trip_id, obs_idx, arc_id). Use only FEATURE_COLUMNS as inputs; do not learn from identifiers or labels. Fit imputation/scaling within each training fold. The observation table retains missing candidates and uncertain labels. All five models must reuse folds.csv.

Defaults are 250 m radius and 30 candidates, selected from a training-only coverage sweep: overall training miss 4.17%, reliable-alignment miss 0.023%. The earlier 50 m / 10 configuration missed 11.74% overall. Uncertain (>50 m aligned distance) observations remain flagged and included; several long training trajectories go far from the supplied reference. Larger candidate sets cost more memory and do not guarantee improved model accuracy. The full export has about 5.2 million training candidates: allow several GB of RAM/disk. No evaluator data is read.

Network interfaces: build_network(trip) provides metric node coordinates, road endpoints, arc IDs, bearings and degrees. build_graph(network) returns a directed scipy CSR graph of shortest parallel-edge lengths; parallel arc identity stays in network arrays. The road spatial index uses full segments, not just nodes/midpoints.

This stage contains no baseline training or HMM/API/UI. The next stage will consume these exact exports.

In [ ]:
from pathlib import Path
import os, subprocess, sys

# In Colab, first clone your Member 1 branch or upload this patch into the clone.
REPO = Path('/content/ML-Hackathon') if Path('/content').exists() else Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
if not (REPO / 'src/member1_pipeline.py').exists():
    raise FileNotFoundError('Set REPO to the cloned repository containing the Member 1 source files.')
os.chdir(REPO)
print(REPO)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', 'numpy==2.3.5', 'pandas==2.2.3', 'scipy==1.17.0', 'scikit-learn==1.8.0', 'pyproj>=3.7,<4', 'pyarrow>=19,<24', 'shapely>=2.1,<3', 'pytest>=8,<10'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'src/member1_checks.py', '-q'], check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'src.member1_pipeline'], check=True)

In [ ]:
import pandas as pd
display(pd.read_csv('results/member1/data/inventory.csv', dtype={'trip_id':str}))

In [ ]:
subprocess.run([sys.executable, '-m', 'src.member1_checks'], check=True)

Reruns reuse caches. If you change radius/K, regenerate all features and retrain downstream models. Do not commit the generated Parquet/cache directory with ordinary Git.